# ESM2 production extraction launcher
This notebook is a thin launcher. It calls the shared manifest-driven runner; no notebook-specific ESM, pooling, or PaRTI implementation is maintained here.

Set the Drive paths below to the uploaded deterministic shards, verified checkpoint, and feature output directory. This notebook is intentionally not executed as part of repository preparation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/lxie41/esm-vcc-features.git'
!git clone --depth 1 $REPO_URL /content/esm-project
%cd /content/esm-project
!pip install -r requirements.txt
import torch
assert torch.cuda.is_available(), 'CUDA GPU required'
print(torch.get_device_name(0), torch.version.cuda, torch.__version__)

In [ ]:
from pathlib import Path
import hashlib, shutil
MODEL_DRIVE = Path('/content/drive/MyDrive/esm2/esm2_t33_650M_UR50D.pt')
MODEL_LOCAL = Path('/content/esm2_t33_650M_UR50D.pt')
EXPECTED_SHA256 = 'EA9D0522B335A8778DEA6535A65301F10208DECE28CD5865482B0B1FC446168C'
shutil.copy2(MODEL_DRIVE, MODEL_LOCAL)
digest = hashlib.sha256(MODEL_LOCAL.read_bytes()).hexdigest().upper()
assert digest == EXPECTED_SHA256, (digest, EXPECTED_SHA256)
print('model checksum verified:', digest)

In [ ]:
# Manifest-driven resumable loop. It skips QC-valid completed Drive outputs.
!PYTHONPATH=src python scripts/run_production_shards.py \
  --manifest /content/drive/MyDrive/esm2/shards/manifest.json \
  --input-root /content/drive/MyDrive/esm2/shards \
  --drive-output-root /content/drive/MyDrive/esm2/features \
  --local-root /content/esm_work \
  --model /content/esm2_t33_650M_UR50D.pt \
  --device cuda \
  --max-tokens 8192 \
  --max-batch-size 16